In [0]:
-- File: databricks/validation/silver_checks.sql
-- Purpose: Validate Silver layer table presence, counts, key quality, and referential integrity.

-- =========================
-- 1) Table existence checks
-- =========================
SHOW TABLES IN silver;

-- =================
-- 2) Row count checks
-- =================
SELECT 'silver.calendar'   AS table_name, COUNT(*) AS row_count FROM silver.calendar
UNION ALL
SELECT 'silver.customers'  AS table_name, COUNT(*) AS row_count FROM silver.customers
UNION ALL
SELECT 'silver.products'   AS table_name, COUNT(*) AS row_count FROM silver.products
UNION ALL
SELECT 'silver.territories' AS table_name, COUNT(*) AS row_count FROM silver.territories
UNION ALL
SELECT 'silver.fact_sales' AS table_name, COUNT(*) AS row_count FROM silver.fact_sales
ORDER BY table_name;

-- =====================================
-- 3) Null checks on critical fact columns
-- =====================================
SELECT
  SUM(CASE WHEN OrderDate IS NULL THEN 1 ELSE 0 END)      AS null_orderdate,
  SUM(CASE WHEN OrderNumber IS NULL THEN 1 ELSE 0 END)    AS null_ordernumber,
  SUM(CASE WHEN OrderLineItem IS NULL THEN 1 ELSE 0 END)  AS null_orderlineitem,
  SUM(CASE WHEN ProductKey IS NULL THEN 1 ELSE 0 END)     AS null_productkey,
  SUM(CASE WHEN CustomerKey IS NULL THEN 1 ELSE 0 END)    AS null_customerkey,
  SUM(CASE WHEN TerritoryKey IS NULL THEN 1 ELSE 0 END)   AS null_territorykey
FROM silver.fact_sales;

-- ==================================
-- 4) Duplicate grain check on fact
-- Grain: (OrderNumber, OrderLineItem)
-- ==================================
SELECT COUNT(*) AS duplicate_grain_groups
FROM (
  SELECT OrderNumber, OrderLineItem, COUNT(*) AS c
  FROM silver.fact_sales
  GROUP BY OrderNumber, OrderLineItem
  HAVING COUNT(*) > 1
) d;

-- ============================
-- 5) Basic business rule checks
-- ============================
SELECT
  SUM(CASE WHEN OrderQuantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity_rows
FROM silver.fact_sales;

-- ===========================
-- 6) Date coverage check (fact)
-- ===========================
SELECT
  MIN(OrderDate) AS min_order_date,
  MAX(OrderDate) AS max_order_date
FROM silver.fact_sales;

-- ==========================================
-- 7) Referential integrity checks from fact
-- ==========================================

-- Missing CustomerKey in silver.customers
SELECT COUNT(*) AS missing_customers
FROM silver.fact_sales f
LEFT JOIN silver.customers c
  ON f.CustomerKey = c.CustomerKey
WHERE c.CustomerKey IS NULL;

-- Missing ProductKey in silver.products
SELECT COUNT(*) AS missing_products
FROM silver.fact_sales f
LEFT JOIN silver.products p
  ON f.ProductKey = p.ProductKey
WHERE p.ProductKey IS NULL;

-- Missing TerritoryKey in silver.territories (territory dim uses SalesTerritoryKey)
SELECT COUNT(*) AS missing_territories
FROM silver.fact_sales f
LEFT JOIN silver.territories t
  ON f.TerritoryKey = t.SalesTerritoryKey
WHERE t.SalesTerritoryKey IS NULL;

-- Missing OrderDate in silver.calendar
SELECT COUNT(*) AS missing_dates
FROM silver.fact_sales f
LEFT JOIN silver.calendar d
  ON f.OrderDate = d.Date
WHERE d.Date IS NULL;

-- ========================================
-- 8) Optional expected-count sanity checks
-- ========================================
-- Update expected values if source changes.
SELECT
  CASE WHEN (SELECT COUNT(*) FROM silver.calendar) = 912 THEN 'PASS' ELSE 'FAIL' END AS calendar_count_check,
  CASE WHEN (SELECT COUNT(*) FROM silver.customers) = 18148 THEN 'PASS' ELSE 'FAIL' END AS customers_count_check,
  CASE WHEN (SELECT COUNT(*) FROM silver.products) = 293 THEN 'PASS' ELSE 'FAIL' END AS products_count_check,
  CASE WHEN (SELECT COUNT(*) FROM silver.territories) = 10 THEN 'PASS' ELSE 'FAIL' END AS territories_count_check,
  CASE WHEN (SELECT COUNT(*) FROM silver.fact_sales) = 56046 THEN 'PASS' ELSE 'FAIL' END AS fact_count_check;
